In [2]:
import sys
sys.path.insert(0, '/Users/sihamchowdhury/Desktop/Projects/finance-dashboards')


# Exploring Labour, Pipeline & Targets Tables

Three tables not yet used in the dashboard:
- `core.fact_sd_timesheet_cost` — people costs by role
- `core.rpt_hubspot_line_report` — pipeline deals
- `core.fact_hubspot_targets` — quarterly revenue targets

Goal: understand grain, quality and how to best use each in the dashboard.

In [3]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()
engine.dispose()
print("Connected")

Connected


---
## 1. Labour — `fact_sd_timesheet_cost`

Daily timesheet entries from Screendragon. Each row is one person × one project × one day.

The SQL classifies roles into three labour types:
- **Research Labor** — VP, SVP, Director, Manager roles
- **Ops Labor** — Field, QA, Programming, Ops, Dashboard roles  
- **Other Labor** — everything else

In [ ]:
# Total labour by type and year
df_labour = pd.read_sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""", engine)

print(df_labour.pivot(index='yr', columns='labor_type', values='labor_cost').round(0))

In [ ]:
# Monthly labour by type for 2025
df_labour_monthly = pd.read_sql("""
    SELECT
        accounting_period_start_date,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost,
        SUM(actual_hours) AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""", engine)

df_labour_monthly["accounting_period_start_date"] = pd.to_datetime(df_labour_monthly["accounting_period_start_date"])
print(f"Rows: {len(df_labour_monthly)}")
df_labour_monthly.head(10)

In [ ]:
# Top roles by cost — useful to validate the classification
pd.read_sql("""
    SELECT
        role_name_at_timesheet_date,
        COUNT(*) as rows,
        SUM(actual_hours) as total_hours,
        SUM(timesheet_external_cost) as total_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 4 DESC
    LIMIT 20
""", engine)

In [ ]:
# Simplified project count check
pd.read_sql("""
    SELECT COUNT(DISTINCT netsuite_project_id) AS timesheet_projects_2025
    FROM core.fact_sd_timesheet_cost
    WHERE EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)


In [ ]:
# 2025 labour summary — total cost, hours, cost per hour by type
df_labour_summary = pd.read_sql("""
    SELECT
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(actual_hours) AS total_hours,
        SUM(timesheet_external_cost) AS total_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 3 DESC
""", engine)

df_labour_summary["cost_per_hour"] = (df_labour_summary["total_cost"] / df_labour_summary["total_hours"]).round(2)
print(df_labour_summary.round(0))

---
## 2. Pipeline — `rpt_hubspot_line_report`

HubSpot deal lines — active pipeline (Closed Won and Closed Lost excluded).

Key questions:
- What stages exist and how much value is at each stage?
- How does pipeline break down by service line and vertical?
- Who are the top owners by pipeline value?

In [ ]:
# Pipeline by stage
df_pipe = pd.read_sql("""
    SELECT
        deal_id,
        deal_pipeline_stage_name,
        COALESCE(vertical, 'Unassigned') AS vertical,
        COALESCE(service_line, 'Unassigned') AS service_line,
        owner_full_name,
        deal_amount_usd AS pipeline_value_usd
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
      AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won', 'closed lost')
      AND deal_amount_usd IS NOT NULL
""", engine)

print(f"Total pipeline rows: {len(df_pipe)}")
print(f"Unique deals: {df_pipe['deal_id'].nunique()}")
print(f"Total pipeline value: ${df_pipe.drop_duplicates('deal_id')['pipeline_value_usd'].sum()/1e6:.1f}M")

print("\nBy stage:")
stage_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("deal_pipeline_stage_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
)
print(stage_summary.round(0))

In [ ]:
# Pipeline by service line
print("By service line:")
sl_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("service_line")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
)
print(sl_summary.round(0))

In [ ]:
# Top 10 owners by pipeline value
print("Top 10 owners:")
owner_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("owner_full_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
    .head(10)
)
print(owner_summary.round(0))

In [ ]:
# Average deal size by stage
print("Average deal size by stage:")
avg_deal = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("deal_pipeline_stage_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .reset_index()
)
avg_deal["avg_deal_size"] = (avg_deal["value"] / avg_deal["deals"]).round(0)
print(avg_deal.sort_values("value", ascending=False).to_string(index=False))

---
## 3. Targets — `fact_hubspot_targets`

Quarterly revenue targets by person and team. Only available from Q1 2025 onwards.

Key questions:
- What teams and people have targets?
- Can we compare targets to actual pipeline won?
- What's the team-level target for 2025?

In [ ]:
# Targets overview
df_targets = pd.read_sql("""
    SELECT
        quarter_start_date,
        quarter_end_date,
        team_primary_name,
        user_first_name || ' ' || user_last_name AS person,
        target_amount_usd
    FROM core.fact_hubspot_targets
    ORDER BY quarter_start_date, team_primary_name
""", engine)

df_targets["quarter_start_date"] = pd.to_datetime(df_targets["quarter_start_date"])
print(f"Total target rows: {len(df_targets)}")
print(f"Date range: {df_targets['quarter_start_date'].min()} to {df_targets['quarter_start_date'].max()}")
print(f"Teams: {df_targets['team_primary_name'].unique().tolist()}")
df_targets.head(10)

In [ ]:
# Annual target by team for 2025
targets_2025 = df_targets[df_targets["quarter_start_date"].dt.year == 2025]

print("2025 targets by team:")
print(
    targets_2025.groupby("team_primary_name")["target_amount_usd"]
    .sum()
    .sort_values(ascending=False)
    .round(0)
)

print(f"\nTotal 2025 target: ${targets_2025['target_amount_usd'].sum()/1e6:.1f}M")

In [ ]:
# Quarterly target totals
print("Quarterly targets (all teams):")
print(
    df_targets.groupby("quarter_start_date")["target_amount_usd"]
    .sum()
    .round(0)
)

In [ ]:
# How does closed won pipeline compare to targets?
# Closed won deals from HubSpot
df_closed_won = pd.read_sql("""
    SELECT
        DATE_TRUNC('quarter', closed_date::timestamp)::date AS quarter_start,
        COUNT(DISTINCT deal_id) AS deals_won,
        SUM(deal_amount_usd) AS revenue_won
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
      AND LOWER(deal_pipeline_stage_name) = 'closed won'
      AND deal_amount_usd IS NOT NULL
      AND closed_date IS NOT NULL
      AND EXTRACT(YEAR FROM closed_date::timestamp) = 2025
    GROUP BY 1
    ORDER BY 1
""", engine)

print("2025 Closed Won by quarter:")
print(df_closed_won)

In [ ]:
# Monthly labour cost trend 2024 vs 2025
df_labour_trend = pd.read_sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        EXTRACT(MONTH FROM accounting_period_start_date)::int AS month_num,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost,
        SUM(actual_hours) AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) IN (2024, 2025)
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""", engine)
print(f"Labour trend rows: {len(df_labour_trend)}")
print(df_labour_trend.groupby(["yr","labor_type"])["labor_cost"].sum().round(0))

---
## Summary — What to add to the dashboard

### Labour (`fact_sd_timesheet_cost`)
Best used as a **standalone labour section** in the dashboard:
- Stacked bar: monthly labour cost by type (Research / Ops / Other)
- Labour cost as % of revenue (labour intensity)
- YoY labour cost trend
- Hours vs cost split (efficiency)

### Pipeline (`rpt_hubspot_line_report`)
Best used as a **pipeline section**:
- Funnel chart by stage (total value at each stage)
- Pipeline by service line (bar)
- Top 10 owners by value (bar)
- Total pipeline value KPI card

### Targets (`fact_hubspot_targets`)
Best used for **actual vs target** tracking:
- Quarterly target vs actual revenue by team
- Attainment % per quarter
- Note: targets only from Q1 2025 — limited to 2025+ comparisons

In [ ]:
import sys
sys.path.insert(0, '/Users/sihamchowdhury/Desktop/Projects/finance-dashboards')

import pandas as pd
from sqlalchemy import text
from src.db.connection import get_engine

engine = get_engine()

# test connection first
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection ok")

In [ ]:
from sqlalchemy import text

# 1. Labour by type and year
print("=== LABOUR BY TYPE & YEAR ===")
with engine.connect() as conn:
    df_lab = pd.read_sql(text("""
        SELECT
            EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
            CASE
                WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%VP%','%SVP%','%Director%','%Manager%'])
                THEN 'Research Labor'
                WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%Field%','%QA%','%Programming%','%Ops%','%Dashboard%'])
                THEN 'Ops Labor'
                ELSE 'Other Labor'
            END AS labor_type,
            SUM(timesheet_external_cost) AS labor_cost,
            SUM(actual_hours) AS total_hours
        FROM core.fact_sd_timesheet_cost
        WHERE accounting_period_is_posted = TRUE
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    """), conn)
print(df_lab.pivot(index='yr', columns='labor_type', values='labor_cost').round(0))

# 2. Pipeline by stage
print("\n=== PIPELINE BY STAGE ===")
with engine.connect() as conn:
    df_pipe = pd.read_sql(text("""
        SELECT
            deal_pipeline_stage_name,
            COUNT(DISTINCT deal_id) AS deals,
            SUM(deal_amount_usd) AS total_value
        FROM core.rpt_hubspot_line_report
        WHERE is_deal_deleted = FALSE
          AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won', 'closed lost')
          AND deal_amount_usd IS NOT NULL
        GROUP BY 1
        ORDER BY 3 DESC
    """), conn)
print(df_pipe.to_string(index=False))

# 3. Targets by team and quarter
print("\n=== TARGETS BY TEAM & QUARTER ===")
with engine.connect() as conn:
    df_tgt = pd.read_sql(text("""
        SELECT
            quarter_start_date,
            team_primary_name,
            SUM(target_amount_usd) AS target
        FROM core.fact_hubspot_targets
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    """), conn)
print(df_tgt.to_string(index=False))
print(f"\nTotal 2025 target: ${df_tgt[df_tgt['quarter_start_date'].astype(str).str.startswith('2025')]['target'].sum()/1e6:.1f}M")